# Ames Housing - Pipeline organizado para modelado

Notebook preparado para usar modelos de regresión (clásicos y PyTorch) con un flujo limpio:
1. Carga
2. Exploración
3. Limpieza e imputación
4. Codificación
5. Split
6. Escalamiento
7. Conversión a tensores

Nota: este dataset es de regresión, por lo que no aplica balanceo de clases.

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FUTURE_REGRESSION_METRICS = {
    'mae': mean_absolute_error,
    'mse': mean_squared_error,
    'r2': r2_score
}
_ = (plt, sns, FUTURE_REGRESSION_METRICS)

## 1. Carga de datos

En esta celda se carga el archivo `train.csv` y se almacena como un DataFrame de pandas para trabajar con la información de forma ordenada.

In [26]:
df = pd.read_csv('Datasets Primer Parcial/1-Ames Housing Dataset/train.csv')
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,534,531363010,20,RL,80.0,9605,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2009,WD,Normal,159000
1,803,906203120,20,RL,90.0,14684,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,6,2009,WD,Normal,271900
2,956,916176030,20,RL,NaN,14375,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,1,2009,COD,Abnorml,137500
3,460,528180130,120,RL,48.0,6472,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2009,WD,Normal,248500
4,487,528290030,80,RL,61.0,9734,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2009,WD,Normal,167000


## 2. Exploración inicial del dataset

Antes de transformar datos, se inspecciona estructura, faltantes, duplicados y comportamiento de `SalePrice`.

In [27]:
print('Dimensiones del dataset:', df.shape)
print('\nPrimeras columnas:')
print(df.columns.tolist()[:20])
print('\nTipos de datos:')
print(df.dtypes.value_counts())
print('\nValores nulos por columna (top 20):')
print(df.isna().sum().sort_values(ascending=False).head(20))
print('\nDuplicados:', df.duplicated().sum())
print('\nResumen de la variable objetivo:')
print(df['SalePrice'].describe())
print('\nMediana de SalePrice:', df['SalePrice'].median())
print('\nEjemplos de variables categóricas:')
cat_sample = df.select_dtypes(include=['object', 'str']).columns[:10]
print(list(cat_sample))

Dimensiones del dataset: (2197, 82)

Primeras columnas:
['Order', 'PID', 'MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area', 'Street', 'Alley', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1', 'Condition 2', 'Bldg Type', 'House Style', 'Overall Qual', 'Overall Cond']

Tipos de datos:
str        43
int64      28
float64    11
Name: count, dtype: int64

Valores nulos por columna (top 20):
Pool QC           2185
Misc Feature      2117
Alley             2054
Fence             1778
Mas Vnr Type      1329
Fireplace Qu      1066
Lot Frontage       362
Garage Qual        122
Garage Yr Blt      122
Garage Cond        122
Garage Finish      122
Garage Type        120
Bsmt Exposure       69
BsmtFin Type 2      68
Bsmt Qual           67
Bsmt Cond           67
BsmtFin Type 1      67
Mas Vnr Area        22
Electrical           1
Bsmt Half Bath       1
dtype: int64

Duplicados: 0

Resumen de la variable objetivo:
count      2197.000000
mean     

## 3. Selección de variables

Se separa la variable objetivo `SalePrice` y se eliminan columnas identificadoras (`Order`, `PID`, `Id`).

In [28]:
target = 'SalePrice'
id_cols = [col for col in ['Id', 'Order', 'PID'] if col in df.columns]
features = df.drop(columns=[target] + id_cols)
features.head()

,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,...,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition
0,20,RL,80.0,9605,Pave,NaN,Reg,Lvl,AllPub,Corner,...,0,0,NaN,NaN,NaN,0,4,2009,WD,Normal
1,20,RL,90.0,14684,Pave,NaN,IR1,Lvl,AllPub,CulDSac,...,0,0,NaN,NaN,NaN,0,6,2009,WD,Normal
2,20,RL,NaN,14375,Pave,NaN,IR1,Lvl,NoSeWa,CulDSac,...,233,0,NaN,NaN,NaN,0,1,2009,COD,Abnorml
3,120,RL,48.0,6472,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,4,2009,WD,Normal
4,80,RL,61.0,9734,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,5,2009,WD,Normal


## 4. Tratamiento de nulos

Se imputan numéricas con mediana y categóricas con `None` para mantener trazabilidad del dato faltante.

In [29]:
num_cols = features.select_dtypes(include=[np.number]).columns
cat_cols = features.select_dtypes(include=['object', 'str']).columns

features[num_cols] = features[num_cols].fillna(features[num_cols].median())
features[cat_cols] = features[cat_cols].fillna('None')

features.isna().sum().sum()

np.int64(0)

## 5. Codificación de categóricas

Se ofrecen dos rutas (`get_dummies` y `OneHotEncoder`) para que puedas comparar rendimiento entre representaciones.

In [30]:
# Version A: get_dummies
features_encoded_dummies = pd.get_dummies(features, columns=cat_cols, drop_first=True)

# Version B: OneHotEncoder
encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
encoded_array = encoder.fit_transform(features[cat_cols])
encoded_cols = encoder.get_feature_names_out(cat_cols)

features_num = features.drop(columns=cat_cols)
features_encoded_ohe = pd.concat(
    [
        features_num.reset_index(drop=True),
        pd.DataFrame(encoded_array, columns=encoded_cols, index=features.index).reset_index(drop=True)
    ],
    axis=1
)

print('Shape con get_dummies:', features_encoded_dummies.shape)
print('Shape con OneHotEncoder:', features_encoded_ohe.shape)
features_encoded_dummies.head()

Shape con get_dummies: (2197, 276)
Shape con OneHotEncoder: (2197, 276)


,MS SubClass,Lot Frontage,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,...,Sale Type_ConLw,Sale Type_New,Sale Type_Oth,Sale Type_VWD,Sale Type_WD,Sale Condition_AdjLand,Sale Condition_Alloca,Sale Condition_Family,Sale Condition_Normal,Sale Condition_Partial
0,20,80.0,9605,7,6,2007,2007,0.0,0.0,0.0,...,False,False,False,False,True,False,False,False,True,False
1,20,90.0,14684,7,7,1990,1991,234.0,485.0,177.0,...,False,False,False,False,True,False,False,False,True,False
2,20,69.0,14375,6,6,1958,1958,541.0,111.0,354.0,...,False,False,False,False,False,False,False,False,False,False
3,120,48.0,6472,9,5,2008,2008,500.0,0.0,0.0,...,False,False,False,False,True,False,False,False,True,False
4,80,61.0,9734,7,5,2004,2004,0.0,241.0,113.0,...,False,False,False,False,True,False,False,False,True,False


## 6. Conversión a arrays

Se elige método de codificación y se generan `X` e `y` en NumPy (`float32`) para modelado.

In [31]:
# Elige: 'dummies' o 'ohe'
encoding_method = 'dummies'

if encoding_method == 'dummies':
    features_encoded = features_encoded_dummies.copy()
elif encoding_method == 'ohe':
    features_encoded = features_encoded_ohe.copy()
else:
    raise ValueError("encoding_method debe ser 'dummies' o 'ohe'")

X = features_encoded.values.astype(np.float32)
y = df[target].values.astype(np.float32).reshape(-1, 1)

print('Metodo de codificacion usado:', encoding_method)
X.shape, y.shape

Metodo de codificacion usado: dummies


((2197, 276), (2197, 1))

## 7. División de datos

Se aplica split 80/20 reproducible con `train_test_split`.

In [ ]:
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train_np.shape, X_test_np.shape, y_train_np.shape, y_test_np.shape

((1758, 276), (439, 276), (1758, 1), (439, 1))

## 8. Escalamiento

Se normaliza con estadísticas del train y se aplica la misma transformación al test.

In [33]:
# Normalización artesanal estilo featureNormalize

def featureNormalize(X):
    X_norm = X.copy()
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)

    # Evita division por cero en columnas constantes
    sigma[sigma == 0] = 1

    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

# Ajuste sobre entrenamiento
X_train_np, mu, sigma = featureNormalize(X_train_np)

# Aplicacion en testeo con los mismos parametros
X_test_np = (X_test_np - mu) / sigma

X_train_np.shape, X_test_np.shape

((1758, 276), (439, 276))

## 9. Conversión a tensores

Se convierten matrices NumPy a tensores PyTorch para entrenamiento.

In [34]:
X_train = torch.from_numpy(X_train_np)
y_train = torch.from_numpy(y_train_np)
X_test = torch.from_numpy(X_test_np)
y_test = torch.from_numpy(y_test_np)

print('Preparación terminada:')
print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_test shape: {y_test.shape}')

Preparación terminada:
X_train shape: torch.Size([1758, 276])
y_train shape: torch.Size([1758, 1])
X_test shape: torch.Size([439, 276])
y_test shape: torch.Size([439, 1])
